In [10]:
from google.colab import drive
import polars as pl  # Using Polars for multi-threaded, memory-efficient data processing as compared to pandas dataframe for large datasets
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the exact path where your files are uploaded
data_dir = '/content/drive/MyDrive/ml-challenge'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 3. Load Source 1,source 2 and source 3 with the explicit tab separator required
# ground_truth contain the actual pairs
source1 = pl.read_csv(f"{data_dir}/train_source1.tsv", separator="\t")
source2 = pl.read_csv(f"{data_dir}/train_source2.tsv", separator="\t")
source3 = pl.read_csv(f"{data_dir}/train_source3.tsv", separator="\t")
ground_truth = pl.read_csv(f"{data_dir}/train_ground_truth.tsv", separator="\t")

print("Source 1 Shape:", source1.shape)
print("Source 2 Shape:", source2.shape)
print("Source 3 Shape:", source3.shape)
print("Ground Truth Shape:", ground_truth.shape)

Source 1 Shape: (2206821, 4)
Source 2 Shape: (5034616, 4)
Source 3 Shape: (5285603, 4)
Ground Truth Shape: (2206821, 2)


In [ ]:
# Check rows where any column has a null/missing value
missing_rows_s1 = source1.filter(
    pl.col("entity_id").is_null() |
    pl.col("business_name").is_null() |
    pl.col("business_address").is_null() |
    pl.col("country").is_null()
)
missing_rows_s2 = source2.filter(
    pl.col("entity_id").is_null() |
    pl.col("business_name").is_null() |
    pl.col("business_address").is_null() |
    pl.col("country").is_null()
)
missing_rows_s3 = source3.filter(
    pl.col("entity_id").is_null() |
    pl.col("business_name").is_null() |
    pl.col("business_address").is_null() |
    pl.col("country").is_null()
)

print(f"Number of rows with missing values in Source 1: {len(missing_rows_s1)}")

print(f"Number of rows with missing values in Source 2: {len(missing_rows_s2)}")

print(f"Number of rows with missing values in Source 3: {len(missing_rows_s3)}")


Number of rows with missing values in Source 1: 0
Number of rows with missing values in Source 2: 168967
Number of rows with missing values in Source 3: 175916


In [ ]:
# Apply fill_null to replace all missing values with empty strings
source1 = source1.fill_null("")
source2 = source2.fill_null("")
source3 = source3.fill_null("")

In [ ]:
# Text Normalization (Lowercasing & Stripping) so that our model treat MUMBAI and mumbai same
def normalize_text_columns(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col("business_name").str.to_lowercase().str.strip_chars(),
        pl.col("business_address").str.to_lowercase().str.strip_chars(),
        pl.col("country").str.to_lowercase().str.strip_chars()
    ])
source1 = normalize_text_columns(source1)
source2 = normalize_text_columns(source2)
source3 = normalize_text_columns(source3)


In [ ]:
# Fixing of all noise patterns
def fix_all_noise_patterns(df: pl.DataFrame) -> pl.DataFrame:
    # 1. Name Variations: Abbreviations & Legal Suffixes
    if "business_name" in df.columns:
        df = df.with_columns(
            pl.col("business_name")
            .str.to_lowercase()
            # Replace ampersands with 'and'
            .str.replace_all(r"&", "and")
            # Standardize corporate suffixes (Corp vs Corporation, Pvt vs Private, Ltd vs Limited, Inc)
            .str.replace_all(r"\bcorp\b", "corporation")
            .str.replace_all(r"\bpvt\b", "private")
            .str.replace_all(r"\bltd\b", "limited")
            .str.replace_all(r"\binc\b", "incorporated")
            .str.strip_chars()
            .alias("business_name")
        )

    # 2. Address Variations: Street Abbreviations (Rd vs Road, St vs Street)
    if "business_address" in df.columns:
        df = df.with_columns(
            pl.col("business_address")
            .str.to_lowercase()
            # Standardize common street type abbreviations
            .str.replace_all(r"\brd\b", "road")
            .str.replace_all(r"\bst\b", "street")
            .str.replace_all(r"\bave\b", "avenue")
            .str.replace_all(r"\bln\b", "lane")
            .str.replace_all(r"\bblvd\b", "boulevard")
            .str.replace_all(r"\bct\b", "court")
            .str.replace_all(r"\bapt\b", "apartment")
            .str.replace_all(r"\bste\b", "suite")
            # US States common short forms
            .str.replace_all(r"\bnc\b", "north carolina")
            .str.replace_all(r"\bma\b", "massachusetts")
            .str.replace_all(r"\bca\b", "california")
            .str.replace_all(r"\boh\b", "ohio")
            .str.replace_all(r"\bny\b", "new york")
            .str.replace_all(r"\btx\b", "texas")
            .str.replace_all(r"\bfl\b", "florida")
            .str.replace_all(r"\bil\b", "illinois")
            .str.replace_all(r"\bwa\b", "washington")
            # Indian States common short forms
            .str.replace_all(r"\bmp\b", "madhya pradesh")
            .str.replace_all(r"\bup\b", "uttar pradesh")
            .str.replace_all(r"\bdel\b", "delhi")
            # Normalize whitespace across component reordering / spacing issues
            .str.replace_all(r"\s+", " ")
            .str.strip_chars()
            .alias("business_address")
        )

    return df

# Apply to all your sources to systematically clean every noise pattern
source1 = fix_all_noise_patterns(source1)
source2 = fix_all_noise_patterns(source2)
source3 = fix_all_noise_patterns(source3)


In [ ]:
# REMOVING STOPWORDS AND UNNECESSARY PUNCTUATION
def advanced_text_cleaning(df: pl.DataFrame, col_name: str = "business_name") -> pl.DataFrame:
    # Keep corporate suffixes! Only remove generic filler words.
    stop_words = {"the", "and", "services", "of", "for","llc"}

    def clean_text(text: str) -> str:
        if not text or not isinstance(text, str):
            return ""

        # 1. Replace ampersand with 'and'
        t = text.replace("&", " and ")

        # 2. Remove punctuation and extra whitespace, keeping alphanumeric and spaces
        t = "".join([ch if ch.isalnum() or ch.isspace() else " " for ch in t])

        # 3. Split into words, filter out stop words and rejoin
        words = [word for word in t.lower().split() if word not in stop_words]

        return " ".join(words)

    return df.with_columns(
        pl.col(col_name)
        .map_elements(clean_text, return_dtype=pl.String)
        .alias(col_name)
    )

source1 = advanced_text_cleaning(source1, "business_name")
source2 = advanced_text_cleaning(source2, "business_name")
source3 = advanced_text_cleaning(source3, "business_name")

print("Advanced text preprocessing with retained suffixes completed!")

Advanced text preprocessing with retained suffixes completed!


In [ ]:
# COUNTINGTHE NUMBER OF ROWS CONTAIN DEVANAGARI LANGUAGE IN BUSSINES_NAME
def count_devanagari_rows(df: pl.DataFrame, col_name: str = "business_name") -> int:
    """
    Counts how many rows contain at least one Devanagari character.
    """
    # Define a helper to check a single string
    def has_devanagari(text: str) -> bool:
        if not text or not isinstance(text, str):
            return False
        return any('\u0900' <= ch <= '\u097F' for ch in text)

    # Filter/count using Polars map_elements
    count = (
        df.select(
            pl.col(col_name)
            .map_elements(has_devanagari, return_dtype=pl.Boolean)
            .sum()
        )
        .item()
    )
    return count

# Check each source
print(f"Source 1 Devanagari count: {count_devanagari_rows(source1, 'business_name')}")
print(f"Source 2 Devanagari count: {count_devanagari_rows(source2, 'business_name')}")
print(f"Source 3 Devanagari count: {count_devanagari_rows(source3, 'business_name')}")
#  SOURCE 2 AND 3 CONTAIN ALMOST HALF NUMBER OF ROWS SO BE CANNOT NEGLECT THEM


Source 1 Devanagari count: 0
Source 2 Devanagari count: 269424
Source 3 Devanagari count: 158003


In [ ]:
# COUNTINGTHE NUMBER OF ROWS CONTAIN DEVANAGARI LANGUAGE IN BUSSINES_NAME
def count_devanagari_rows(df: pl.DataFrame, col_name: str = "business_address") -> int:
    """
    Counts how many rows contain at least one Devanagari character.
    """
    # Define a helper to check a single string
    def has_devanagari(text: str) -> bool:
        if not text or not isinstance(text, str):
            return False
        return any('\u0900' <= ch <= '\u097F' for ch in text)

    # Filter/count using Polars map_elements
    count = (
        df.select(
            pl.col(col_name)
            .map_elements(has_devanagari, return_dtype=pl.Boolean)
            .sum()
        )
        .item()
    )
    return count

# Check each source
print(f"Source 1 Devanagari count: {count_devanagari_rows(source1, 'business_address')}")
print(f"Source 2 Devanagari count: {count_devanagari_rows(source2, 'business_address')}")
print(f"Source 3 Devanagari count: {count_devanagari_rows(source3, 'business_address')}")
#  SOURCE 2 AND 3 CONTAIN ALMOST HALF NUMBER OF ROWS SO BE CANNOT NEGLECT THEM

Source 1 Devanagari count: 0
Source 2 Devanagari count: 277523
Source 3 Devanagari count: 275861


In [ ]:

# Comprehensive Devanagari to Latin phonetic mapping dictionary
DEVA_TO_LATIN = {
    'अ': 'a', 'आ': 'aa', 'इ': 'i', 'ई': 'ee', 'उ': 'u', 'ऊ': 'oo', 'ऋ': 'ri', 'ए': 'e', 'ऐ': 'ai', 'ओ': 'o', 'औ': 'au',
    'क': 'ka', 'ख': 'kha', 'ग': 'ga', 'घ': 'gha', 'ङ': 'nga',
    'च': 'cha', 'छ': 'chha', 'ज': 'ja', 'झ': 'jha', 'ञ': 'nya',
    'ट': 'ta', 'ठ': 'tha', 'ड': 'da', 'ढ': 'dha', 'ण': 'na',
    'त': 'ta', 'थ': 'tha', 'द': 'da', 'ध': 'dha', 'न': 'na',
    'प': 'pa', 'फ': 'pha', 'ब': 'ba', 'भ': 'bha', 'म': 'ma',
    'य': 'ya', 'र': 'ra', 'ल': 'la', 'व': 'va', 'श': 'sha', 'ष': 'sha', 'स': 'sa', 'ह': 'ha',
    '़': '', 'ा': 'aa', 'ि': 'i', 'ी': 'ee', 'ु': 'u', 'ू': 'oo', 'ृ': 'ri', 'े': 'e', 'ै': 'ai', 'ो': 'o', 'ौ': 'au',
    '्': '', 'ं': 'n', 'ः': 'h',
    # Common Devanagari numerals mapping
    '०': '0', '१': '1', '२': '2', '३': '3', '४': '4', '५': '5', '६': '6', '७': '7', '८': '8', '९': '9'
}

def transliterate_devanagari_to_latin(text: str) -> str:
    if not text or not isinstance(text, str):
        return ""
    # Map characters if they exist in the dictionary, otherwise keep original
    return "".join([DEVA_TO_LATIN.get(ch, ch) for ch in text])

def clean_and_transliterate_columns(df: pl.DataFrame, columns: list[str]) -> pl.DataFrame:
    """
    Applies Devanagari-to-Latin transliteration across multiple specified columns.
    """
    exprs = [
        pl.col(col)
        .map_elements(transliterate_devanagari_to_latin, return_dtype=pl.String)
        .alias(col)
        for col in columns
    ]
    return df.with_columns(exprs)

# Target columns to clean
target_columns = ["business_name", "business_address"]

# Apply to your sources (Source 2 and Source 3 contain the regional text)
source1 = clean_and_transliterate_columns(source1, target_columns)
source2 = clean_and_transliterate_columns(source2, target_columns)
source3 = clean_and_transliterate_columns(source3, target_columns)

print("Business names and addresses successfully cleaned and transliterated across all sources!")

Business names and addresses successfully cleaned and transliterated across all sources!


In [ ]:
# CONVERSION OF DIFFERENT FORMANT OF INDIA AND US IN SINGLE FORMAT
def standardize_countries(df: pl.DataFrame) -> pl.DataFrame:
    # Check if 'country' column exists
    if "country" in df.columns:
        return df.with_columns(
            pl.col("country")
            .str.to_lowercase()
            .str.strip_chars()
            .replace({
                "united states": "us",
                "u.s.a.": "us",
                "usa": "us",
                "bharat": "india",
                "in": "india",
                "france": "france"
            }, default=pl.col("country"))
            .alias("country")
        )
    return df

# Apply to your sources
source1 = standardize_countries(source1)
source2 = standardize_countries(source2)
source3 = standardize_countries(source3)


/tmp/ipykernel_1657/3514526420.py:9: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  .replace({


In [ ]:
# COUNTING NUMBER OF DUBLICATES DATA BASED ON SOURCE ID
def count_duplicate_ids(df: pl.DataFrame, df_name: str):
    duplicates = (
        df.group_by("entity_id")
        .len()
        .filter(pl.col("len") > 1)
    )
    print(f"Number of duplicate entity_ids in {df_name}: {duplicates.height}")
    if duplicates.height > 0:
        print(duplicates.head())

# Check across your sources
count_duplicate_ids(source1, "Source 1")
count_duplicate_ids(source2, "Source 2")
count_duplicate_ids(source3, "Source 3")

Number of duplicate entity_ids in Source 1: 0
Number of duplicate entity_ids in Source 2: 0
Number of duplicate entity_ids in Source 3: 0


In [ ]:
# CALCULATING THE AVERAGE LENGTH OF BUSSINES_NAME IN EACH COLUMN
avg_length = source1.select(
    pl.col("business_name").str.len_chars().mean()
).item()
avg_length = source2.select(
    pl.col("business_name").str.len_chars().mean()
).item()
avg_length = source3.select(
    pl.col("business_name").str.len_chars().mean()
).item()

print(f"Average business name length: {avg_length:.2f} characters")
print(f"Average business name length: {avg_length:.2f} characters")
print(f"Average business name length: {avg_length:.2f} characters")

Average business name length: 25.41 characters
Average business name length: 25.41 characters
Average business name length: 25.41 characters


In [ ]:
# DEFINING BLOCKING_KEY
def add_robust_blocking_key(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        # 1. Core token
        # we use first four character of each bussiness_name after sorting (TATASTEEL VS STEELTATA)
        core_token=pl.col("business_name")
            .str.split(" ")
            .list.eval(pl.element().filter(pl.element().str.len_chars() >= 3))
            .list.eval(pl.element().sort())
            .list.first()
            .str.slice(0, 4)
            .fill_null("unknown"),

        # 2. Address numeric prefix
        # we use last 5 digit pin code to make pair
        address_prefix=pl.col("business_address")
            .str.extract(r"(\d{5})", 1)
            .fill_null("99999"),

        # 3. Alphabetically sorted Initials Key
        # we take first character of each word in bussines_address after sorting
        initials_key=pl.col("business_name")
            .str.split(" ")
            .list.eval(pl.element().filter(pl.element().str.len_chars() > 0))
            .list.eval(pl.element().sort())
            .list.eval(pl.element().str.slice(0, 1))
            .list.eval(pl.element().str.concat(""))
            .list.first()
            .fill_null("unk"),

        # 4. Name length bucket based on your 25-char average (S = Short, L = Long)
        # based on OVERALL average lenths of bussiness_name
        name_len_bin=pl.when(pl.col("business_name").str.len_chars() < 25).then(pl.lit("S")).otherwise(pl.lit("L"))
    ).with_columns(
        # makes single blocking key help to divide the input data
        blocking_key=(
            pl.col("country") + "_" +
            pl.col("core_token") + "_" +
            pl.col("initials_key") + "_" +
            pl.col("name_len_bin") + "_" +
            pl.col("address_prefix")
        )
    )

# Re-apply to all sources
source1 = add_robust_blocking_key(source1)
source2 = add_robust_blocking_key(source2)
source3 = add_robust_blocking_key(source3)

# Clean up unwanted columns
source1 = source1.drop(["first_token", "blocking_key_2"], strict=False)
source2 = source2.drop(["first_token", "blocking_key_2"], strict=False)
source3 = source3.drop(["first_token", "blocking_key_2"], strict=False)

/tmp/ipykernel_1657/4220013027.py:27: DeprecationWarning: `str.concat` is deprecated; use `str.join` instead. Note also that the default `delimiter` for `str.join` is an empty string, not a hyphen.
  .list.eval(pl.element().str.concat(""))


In [ ]:
# we store the file in parquet format in my drive so it take less space in our drive
source1.write_parquet("source1_blocked.parquet")
source2.write_parquet("source2_blocked.parquet")
source3.write_parquet("source3_blocked.parquet")

print("All three blocked sources successfully saved as Parquet files locally!")